In [173]:
import warnings
warnings.filterwarnings('ignore')
import requests
import datetime as dt
import numpy as np
import pandas as pd
import yfinance as yf

# Algorithmic Trading

## Introduction

Algorithmic trading means using computers to make investment decisions. There are many different types of algorithmic trading. The main difference is their speed of execution.

Some of the main players in the algorithmic trading landscape are Renaissance Technologies, AQR Capital Management, Citadel Securities.

Python is the most popular programming language for algorithmic trading. However, Python is slow. This means that it is often used as a "glue language" to trigger code that runs in other languages. One example of this is the NumPy library for Python, which we'll be using in this course. NumPy is the most popular Python library for performing numerical computing. Although it's written for use in Python underlying functionality is written in faster language.

The process of running a quantitative investing strategy can be broken down into the following steps:
1. Collect data.
2. Develop a hypothesis for a strategy.
3. Backtest that strategy.
4. Implement the strategy in production.

Because this is an introductory course, it will differ from production algorithmic trading in 3
major ways:
1. We'll be using random data.
2. We will not be executing trades.
3. We will be saving recommended trades in files.

In this course, we'll be using the IEX Cloud API to gather stock market data to make investment decisions.

## Project 1: Equal-Weight S&P 500 Strategy

The S&P 500 is the world's most popular stock market index. Many investment funds are benchmarked to the S&P 500. This means that they seek to replicate the performance of this index by owning all the stocks that are held in the index.

One of the most important characteristics of the S&P500 is that it is market capitalization-weighted. This means that larger companies get a correspondingly larger weight in the index. In the first project of this course, we will build an alternative version of the S&P 500 Index fund where each company has the same weight.

In [92]:
def get_sp500_tickers() -> list:
    WIKIPEDIA_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    stocks_df = pd.read_html(WIKIPEDIA_URL)[0]
    sp500_tickers = stocks_df.Symbol.to_list()
    return sp500_tickers

In [132]:
def get_latest_stock_data(tickers_list: list) -> pd.DataFrame:
    today = dt.datetime.today()
    yesterday = today - dt.timedelta(days=1)

    stock_df = yf.download(tickers_list, start=yesterday, end=today)[["Adj Close"]]
    
    # Pivot table and renaming
    stock_df = stock_df.stack().reset_index(level=1, names=["", "Ticker"]).sort_values(by="Ticker").reset_index(drop=True).rename_axis(None, axis=1)
    stock_df.rename(columns = {
        "Ticker": "ticker",
        "Adj Close": "price",
    }, inplace=True)
    
    return stock_df 

In [90]:
def equal_weight_calcultor(df: pd.DataFrame, portfolio_size: int = 1000) -> pd.DataFrame:    
    position_size = portfolio_size/df.shape[0]
    df["number_of_shares_to_buy"] = np.floor(portfolio_size/df["price"]).astype(int)
    return df

In [93]:
def sp500_equal_weight_strategy(portfolio_size: int = 1000) -> pd.DataFrame:
    sp500_tickers = get_sp500_tickers()
    sp500_df = get_latest_stock_data(tickers_list=sp500_tickers)
    sp500_equal_weight_df = equal_weight_calcultor(df=sp500_df, portfolio_size=1000)
    return sp500_df

df = sp500_equal_weight_strategy()
df

[*****************     35%%                      ]  177 of 503 completed

$BF.B: possibly delisted; No price data found  (1d 2024-05-24 09:37:17.704885 -> 2024-05-25 09:37:17.704885)


[*********************100%%**********************]  503 of 503 completed

2 Failed downloads:
['BF.B']: YFPricesMissingError('$%ticker%: possibly delisted; No price data found  (1d 2024-05-24 09:37:17.704885 -> 2024-05-25 09:37:17.704885)')
['BRK.B']: YFTzMissingError('$%ticker%: possibly delisted; No timezone found')


,ticker,price,number_of_shares_to_buy
0,A,150.660004,6
1,AAL,13.840000,72
2,AAPL,189.979996,5
3,ABBV,157.059998,6
4,ABNB,144.470001,6
...,...,...,...
496,XYL,144.250000,6
497,YUM,137.649994,7
498,ZBH,116.410004,8
499,ZBRA,327.000000,3
